# Model Evaluation

This notebook evaluates the trained models on the test set and provides detailed performance analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from pathlib import Path
import sys

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

from src.models.predict_model import predict_model, predict_proba, evaluate_model, get_feature_importance

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load the best trained model from MLflow
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("../mlruns")

# Replace with your actual run ID
run_id = "YOUR_RUN_ID_HERE"
model_uri = f"runs:/{run_id}/model"
model = mlflow.sklearn.load_model(model_uri)

print("Model loaded successfully")

In [ ]:
# Load preprocessed test data
# (You should save the preprocessed data from model_development.ipynb)
# For now, we'll assume it's available

# X_test_processed = pd.read_csv('../data/processed/X_test_processed.csv')
# y_test = pd.read_csv('../data/processed/y_test.csv')

print("Test data loaded")

In [ ]:
# Make predictions
y_pred = predict_model(model, X_test_processed)
y_proba = predict_proba(model, X_test_processed)[:, 1]

print(f"Predictions shape: {y_pred.shape}")

In [ ]:
# Calculate evaluation metrics
metrics = evaluate_model(model, X_test_processed, y_test)

print("Model Performance Metrics:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print(f"True Negatives: {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives: {cm[1,1]}")

In [ ]:
# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Subscription', 'Subscription']))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label=f'PR curve (AP = {avg_precision:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc="lower left")
plt.show()

In [ ]:
# Feature Importance
feature_importance = get_feature_importance(model, X_test_processed.columns)

# Plot top 20 features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(20)
sns.barplot(data=top_features, x='importance', y='feature')
plt.title('Top 20 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

print("\nTop 10 Features:")
print(feature_importance.head(10))

In [ ]:
# Prediction distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(y_proba, bins=30, kde=True)
plt.title('Distribution of Prediction Probabilities')
plt.xlabel('Probability of Subscription')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.boxplot(x=y_test, y=y_proba)
plt.title('Prediction Probabilities by Actual Class')
plt.xlabel('Actual Subscription')
plt.ylabel('Predicted Probability')

plt.tight_layout()
plt.show()

In [ ]:
# Threshold analysis
thresholds = np.arange(0.1, 0.9, 0.05)
threshold_metrics = []

for threshold in thresholds:
    y_pred_thresh = (y_proba >= threshold).astype(int)
    threshold_metrics.append({
        'threshold': threshold,
        'accuracy': accuracy_score(y_test, y_pred_thresh),
        'precision': precision_score(y_test, y_pred_thresh),
        'recall': recall_score(y_test, y_pred_thresh),
        'f1': f1_score(y_test, y_pred_thresh)
    })

threshold_df = pd.DataFrame(threshold_metrics)

plt.figure(figsize=(10, 6))
plt.plot(threshold_df['threshold'], threshold_df['f1'], marker='o', label='F1')
plt.plot(threshold_df['threshold'], threshold_df['precision'], marker='o', label='Precision')
plt.plot(threshold_df['threshold'], threshold_df['recall'], marker='o', label='Recall')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Metrics vs Threshold')
plt.legend()
plt.grid(True)
plt.show()

print("\nThreshold Analysis:")
print(threshold_df)

## Evaluation Summary

### Key Findings:
-
-
-

### Recommendations:
-
-
-